In [ ]:
import os
from pathlib import Path

from ultralytics.cfg import get_cfg
from ultralytics.data import build_dataloader, build_yolo_dataset
from ultralytics.data.utils import check_det_dataset

# Run with the kernel in the repo root or training/ (install requirements.txt first).
repo = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()
data_yaml = repo / os.environ.get("DEEPFASHION2_DATA", "data/deepfashion2-subset-2000-seed42/data.yaml")
data = check_det_dataset(data_yaml, autodownload=False)
cfg = get_cfg(overrides=dict(task="segment", imgsz=320, overlap_mask=False,
                             mosaic=0.0, scale=0.0, translate=0.0))
for split in ("train", "val"):
    dataset = build_yolo_dataset(cfg, data[split], 4, data, mode=split, rect=(split == "val"))
    # For a frozen subset, audit the complete file list and garment classes before batching.
    listing = Path(data[split])
    if listing.is_file():
        expected = {(listing.parent / line).resolve() for line in listing.read_text().splitlines()}
        assert {Path(f).resolve() for f in dataset.im_files} == expected
        for label in dataset.labels:
            txt = data_yaml.parent / 'labels' / split / (Path(label['im_file']).stem + '.txt')
            disk_classes = sorted(int(row.split()[0]) for row in txt.read_text().splitlines())
            assert sorted(label['cls'].flatten().tolist()) == disk_classes
            assert len(label['segments']) == len(disk_classes)
    loader = build_dataloader(dataset, batch=4, workers=0, shuffle=False, device="cpu")
    batch = next(iter(loader))
    masks = batch["masks"]
    assert dataset.augment == (split == "train")
    assert len(batch["cls"]) == len(masks) > 0 and masks.flatten(1).any(1).all()
    print(f"{split}: {len(batch['img'])} images, {len(masks)} garment masks, augment={dataset.augment}")

In [ ]:
import os
import sys
from pathlib import Path

repo = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from preprocessing.check_training_augmentation import check_training_batches

data_yaml = repo / os.environ.get("DEEPFASHION2_DATA", "data/deepfashion2-subset-2000-seed42/data.yaml")
train_batch, val_batch = check_training_batches(data_yaml)
print(train_batch["img"].shape, val_batch["img"].shape)